In [ ]:
# 4) Analise a evolução mensal do volume de pedidos ao longo de 2023 e 2024. Há sazonalidade? Identifique picos e quedas e formule uma hipótese para cada padrão encontrado.

import sys
if hasattr(sys.stdout, "reconfigure"):
    try:
        sys.stdout.reconfigure(encoding="utf-8")
    except Exception:
        pass
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

DATA_DIR = Path('../data')
DARK_BG, GRID_CLR, SPINE_CLR, BLUE = '#121212', '#333333', '#555555', '#00D2FF'

plt.rcParams.update({
    'figure.facecolor': DARK_BG, 'axes.facecolor': DARK_BG,
    'axes.edgecolor': SPINE_CLR, 'axes.labelcolor': 'white',
    'xtick.color': 'white', 'ytick.color': 'white', 'text.color': 'white',
    'grid.color': GRID_CLR, 'grid.linestyle': '--', 'grid.alpha': 0.7,
})

STATUS_LABEL = {
    'em_transito': 'Em Trânsito', 'entregue': 'Entregue', 'cancelado': 'Cancelado',
    'aguardando_pagamento': 'Aguardando Pagamento', 'devolvido': 'Devolvido',
}

def load_clean(tabela: str) -> pd.DataFrame:
    path = DATA_DIR / f'{tabela}_limpo.csv'
    if not path.exists(): raise FileNotFoundError("Erro: Rode o script ETL (Q6) primeiro.")
    return pd.read_csv(path)

df_pedidos  = load_clean('pedidos')
df_clientes = load_clean('clientes')

df_pedidos['data_pedido'] = pd.to_datetime(df_pedidos['data_pedido'], errors='coerce')
df_2324 = df_pedidos[df_pedidos['data_pedido'].dt.year.isin([2023, 2024])].copy()
df_2324['ano_mes']     = df_2324['data_pedido'].dt.to_period('M')
df_2324['ano_mes_str'] = df_2324['data_pedido'].dt.strftime('%Y-%m')

evolucao = df_2324.groupby('ano_mes').size()
x_labels = evolucao.index.astype(str)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(x_labels, evolucao.values, color=BLUE, marker='o', linewidth=3, markersize=8)

for i, v in enumerate(evolucao.values):
    ax.annotate(str(v), (x_labels[i], v), textcoords='offset points', xytext=(0, 10), ha='center', fontsize=9, fontweight='bold')

ax.set_title('Evolução Mensal do Volume de Pedidos (2023–2024)', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Mês e Ano', fontweight='bold', labelpad=15)
ax.set_ylabel('Volume de Pedidos', fontweight='bold', labelpad=15)
ax.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# ── Validação de Hipóteses ──
df_nov = df_2324[df_2324['ano_mes_str'].isin(['2023-11', '2024-11'])].copy()

print("=" * 70)
print("HIPÓTESE 1: ESTRATÉGIA DE DESCONTOS")
cupons = df_nov.groupby('ano_mes_str')['cupom_desconto'].value_counts(normalize=True).unstack() * 100
display(cupons.fillna(0).round(1).style.format("{:.1f}%"))
print("→ Penetração de cupons similar entre nov/23 e nov/24: HIPÓTESE REFUTADA.")

print("\n" + "=" * 70)
print("HIPÓTESE 2: CONCENTRAÇÃO B2B ATÍPICA EM NOV/23")
df_nov_cli = df_nov.merge(df_clientes[['id', 'segmento']], left_on='cliente_id', right_on='id', how='inner')
seg_dist = df_nov_cli.groupby('ano_mes_str')['segmento'].value_counts(normalize=True).unstack() * 100
display(seg_dist.fillna(0).round(1).style.format("{:.1f}%"))
print("→ Participação B2B similar: HIPÓTESE REFUTADA.")

print("\n" + "=" * 70)
print("HIPÓTESE 3: RUPTURA OPERACIONAL (TAXA DE CANCELAMENTO)")
status_dist = df_nov.groupby('ano_mes_str')['status'].value_counts(normalize=True).unstack() * 100
status_dist = status_dist.rename(columns=STATUS_LABEL)
display(status_dist.fillna(0).round(1).style.format("{:.1f}%"))
print("→ Taxa de cancelamento idêntica: HIPÓTESE REFUTADA.")

print("\n" + "=" * 70)
print("HIPÓTESE 4: DUPLICAÇÃO DE DADOS (ERRO DE SISTEMA)")
df_nov23 = df_nov[df_nov['ano_mes_str'] == '2023-11']
n_dup    = df_nov23.duplicated(subset=['id']).sum()

print(f"Total de Pedidos em Nov/23:    {len(df_nov23):,}")
print(f"Pedidos DUPLICADOS em Nov/23:  {n_dup:,}")

if n_dup > 0:
    print(f"\nALERTA: {n_dup} pedidos duplicados! Pico pode ser artificial.")
    print("   → O pico de Nov/23 é um artefato de dados — HIPÓTESE CONFIRMADA.")
else:
    print("\nBase limpa. Nenhuma duplicação detectada.")